# Doğal Dil İşleme + Karar Bilimi (Natural Language Processing + Decision Science)

👩🏻‍🏫 Bu görevde şunları bir araya getireceğiz:
* 🗣 Doğal Dil İşleme (Natural Language Processing)
* 📊 Karar Bilimi (Decision Science)

🎯 Amaç, Olist üzerindeki ürünlerin ve satıcıların **olumsuz (kötü) yorumlarını** anlamaktır.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Data Manipulation
import numpy as np
import pandas as pd
pd.set_option("display.max_columns",None)

# Machine Learning
from sklearn.pipeline import make_pipeline

# Language Processing
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk
import string
import unidecode as unidecode

# Vectorizers and NLP Models
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

🕵🏻‍♂️ Olist’in CEO’su [Tiago Dalvi](https://www.linkedin.com/in/tiagodalvi/)’nin senden yorumları okuyup anlamanı istediğini hayal et.

- Müşteriler siparişlerini **1**, **2** veya **3** puanla değerlendirdiklerinde ne söylediler?
- En sık karşılaşılan olumsuz yorumlar neler?
    - En kötü puanlanan ürünler hakkında?
    - En kötü puanlanan satıcılar hakkında?


## (0) Kurulum 🔨

Öncelikle, Olist incelemeleriyle ilgili tüm bilgileri içeren DataFrame'i yükleyeceğiz!

In [3]:
df = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/olist_reviews.csv")

In [4]:
df.head()

,review_id,length_review,review_score,order_id,product_category_name,full_review
0,e64fb393e7b32834bb789ff8bb30750e,37,5,658677c97b385a9be170737859d3511b,ferramentas_jardim,Recebi bem antes do prazo estipulado.
1,f7c4243c7fe1938f181bec41a392bdeb,100,5,8e6bfb81e283fa7e4f11123a3fb894f1,esporte_lazer,Parabéns lojas lannister adorei comprar pela ...
2,8670d52e15e00043ae7de4c01cc2fe06,174,4,b9bf720beb4ab3728760088589c62129,eletroportateis,recomendo aparelho eficiente. no site a marca ...
3,4b49719c8a200003f700d3d986ea1a19,45,4,9d6f15f95d01e79bd1349cc208361f09,beleza_saude,"Mas um pouco ,travando...pelo valor ta Boa.\r\n"
4,3948b09f7c818e2d86c9a546758b2335,56,5,e51478e7e277a83743b6f9991dbfa3fb,informatica_acessorios,"Super recomendo Vendedor confiável, produto ok..."


## (2) Metin Temizleme (Text Cleaning)

🧹 `cleaning(sentence)` işlevini oluşturun ve yorumlara uygulayın. **NLTK'da Portekizce lemmatizer bulunmadığını unutmayın** (genellikle bulunmaz, ancak `nltk.stem.RSLPStemmer` kök bulucu vardır).

In [5]:
import nltk
import re
import string
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer

# Portekizce kaynakları indir
nltk.download('stopwords')
nltk.download('rslp') # Portekizce Stemmer için gerekli

def cleaning(sentence):
    # 1. Küçük harfe çevirme
    sentence = sentence.lower()
    
    # 2. Noktalama işaretlerini ve sayıları kaldırma
    sentence = re.sub(f"[{re.escape(string.punctuation)}]", " ", sentence)
    sentence = re.sub(r'\d+', '', sentence)
    
    # 3. Tokenization (Kelimelere ayırma)
    words = sentence.split()
    
    # 4. Stopwords kaldırma ve Stemming (Kök bulma)
    pt_stopwords = set(stopwords.words('portuguese'))
    stemmer = RSLPStemmer()
    
    # Kelime köküne indir ve gereksiz kelimeleri at
    cleaned_words = [stemmer.stem(w) for w in words if w not in pt_stopwords]
    
    return " ".join(cleaned_words)

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/aybukealtuntas/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package rslp to
[nltk_data]     /Users/aybukealtuntas/nltk_data...
[nltk_data]   Package rslp is already up-to-date!


In [6]:
# DataFrame'e uygulama (full_review sütununa)
df['cleaned_review'] = df['full_review'].apply(cleaning)

# Sonucu kontrol etme
print(df[['full_review', 'cleaned_review']].head())

                                         full_review  \
0              Recebi bem antes do prazo estipulado.   
1   Parabéns lojas lannister adorei comprar pela ...   
2  recomendo aparelho eficiente. no site a marca ...   
3    Mas um pouco ,travando...pelo valor ta Boa.\r\n   
4  Super recomendo Vendedor confiável, produto ok...   

                                      cleaned_review  
0                         receb bem ant praz estipul  
1  parabém loj lannist ador compr internet segur ...  
2  recom aparelh efici sit marc aparelh impress d...  
3                               pouc trav val ta boa  
4     sup recom vend confi produt ok entreg ant praz  


## (3) Kötü yorumların analizi

### (3.1) Düşük inceleme puanlarına sahip veri kümesi

😱 1 ile 3 arasında puan alan yorumların oranı nedir? 

In [7]:
# 1, 2 ve 3 puan alanların sayısını toplam satır sayısına bölüyoruz
low_scores = df[df['review_score'].isin([1, 2, 3])]
ratio = (len(low_scores) / len(df)) * 100

print(f"1-3 arası puan alan yorumların oranı: %{ratio:.2f}")

1-3 arası puan alan yorumların oranı: %26.72


🕵🏻‍♂️ Bu yorumlara odaklanalım...

In [8]:
# 1, 2 ve 3 puanlı yorumları filtrele
df_low = df[df['review_score'] <= 3].copy()

# En çok kullanılan kelimeleri görmek için bir göz atalım
print(df_low[['review_score', 'full_review', 'cleaned_review']].head())

    review_score                                        full_review  \
14             1   recebi somente 1 controle Midea Split ESTILO....   
23             3   Eu comprei duas unidades e só recebi uma e ag...   
24             3   Produto bom, porém o que veio para mim não co...   
25             1               Produto muito inferior, mal acabado.   
28             3                                   Entrega no prazo   

                                       cleaned_review  
14  receb soment control mide split estil falt con...  
23                    compr dua unidad receb agor faç  
24         produt bom porém vei mim condiz fot anúnci  
25                             produt inferi mal acab  
28                                        entreg praz  


### (3.2) Vektörleştirme

🔡 ➡️ 🔢 Metinlerini vektörleştir.

- **Bigram**’leri (iki kelimelik ifadeler) mutlaka hesaba kat.
- Çok sık geçen kelimeleri çıkarmak için `max_df = 0.75` ayarla.
- Spoiler uyarısı: Sonunda **20.000+** kelimeye ulaşacaksın…  
  Bu challenge için sadece `max_features = 5000` ile sınırla.

In [55]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# 1. TF-IDF Vektörleştiriciyi tanımlıyoruz
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),    # Hem tekli kelimeler hem de Bigram'lar (ikili gruplar)
    max_df=0.75,           # Dokümanların %75'inden fazlasında geçen kelimeleri ele (çok genel kelimeler)
    max_features=5000      # Sözlüğü en önemli 5000 kelime/ifade ile sınırla
)

X_tfidf = vectorizer.fit_transform(df_low['cleaned_review'].fillna('').astype('U'))

tfidf_df = pd.DataFrame(X_tfidf.toarray(), columns=vectorizer.get_feature_names_out())

print(f"Matris boyutu: {tfidf_df.shape}")
print("İlk 5 sütun (bazı özellikler):", tfidf_df.columns[:5].tolist())

Matris boyutu: (9658, 5000)
İlk 5 sütun (bazı özellikler): ['abaix', 'abert', 'abert rasg', 'aborrec', 'abr']


### (3.3) LDA

🕵🏻‍♂️ LDA'yı uyarlayın:
- `n_components = 3` seçin
- *.transform()* ile Konuların Belge Karışımını gösterin
- *.components_* ile Konu Karışımını gösterin

In [56]:
from sklearn.decomposition import LatentDirichletAllocation

# 1. Modeli tanımla ve eğit (3 Ana Konu)
lda_model = LatentDirichletAllocation(n_components=3, random_state=42)
lda_model.fit(X_tfidf)

LatentDirichletAllocation(n_components=3, random_state=42)

#### Belge Karışımı (Konular için)

In [57]:
# 2. .transform() ile "Konuların Belge Karışımı" (Document-Topic Distribution)
# Her bir yorumun hangi konuya ne kadar ait olduğunu gösterir
doc_topic_mixture = lda_model.transform(X_tfidf)

👉 Her inceleme için en önemli konuyu rapor edelim

In [58]:
# 3. .components_ ile "Kelime-Konu Karışımı" (Topic-Word Distribution)
# Her bir konunun hangi kelimelerden oluştuğunu gösterir
topic_word_mixture = lda_model.components_

#### Konu Karışımı (Kelimeler için)

In [59]:
print(f"Belge Karışımı Şekli: {doc_topic_mixture.shape}") 
print(f"Konu Karışımı Şekli: {topic_word_mixture.shape}")

Belge Karışımı Şekli: (9658, 3)
Konu Karışımı Şekli: (3, 5000)


In [60]:
# Her konunun en önemli 10 kelimesini/ifadesini listeleyelim
words = vectorizer.get_feature_names_out()

for i, topic in enumerate(lda_model.components_):
    top_words_idx = topic.argsort()[-10:] # En yüksek ağırlıklı son 10 kelimenin indeksi
    top_words = [words[idx] for idx in top_words_idx]
    print(f"Konu #{i+1}: {' | '.join(top_words)}")

Konu #1: ped | apen | nao | receb produt | compr | aind | produt | entreg | bom | receb
Konu #2: qual | fot | receb | defeit | compr | entreg | praz | vei | gost | produt
Konu #3: cor | demor | cheg | boa | recom | entreg | compr | err | vei | produt


#### Konular

🎁 Size bazı yardımcı fonksiyonlar sağladık:
- `topic_word`: Tek bir konu (topic) için en önemli kelimeleri ve ağırlıklarını döndürür
- `print_topics`: LDA tarafından bulunan farklı konuları, en önemli kelimeleriyle birlikte yazdırır

In [61]:
def topic_word(vectorizer, model, topic, topwords, with_weights = True):
    topwords_indexes = topic.argsort()[:-topwords - 1:-1]
    if with_weights == True:
        topwords = [(vectorizer.get_feature_names_out()[i], round(topic[i],2)) for i in topwords_indexes]
    if with_weights == False:
        topwords = [vectorizer.get_feature_names_out()[i] for i in topwords_indexes]
    return topwords

In [62]:
def print_topics(vectorizer, model, topwords):
    for idx, topic in enumerate(model.components_):
        print("-"*20)
        print("Topic %d:" % (idx))
        print(topic_word(vectorizer, model, topic, topwords))

🕵🏻‍♂️ Konuları en çok kullanılan kelimelerle birlikte yazdırın:

In [63]:
print_topics(vectorizer,lda_model,10)

--------------------
Topic 0:
[('receb', 250.08), ('bom', 213.24), ('entreg', 169.16), ('produt', 163.1), ('aind', 130.84), ('compr', 120.72), ('receb produt', 111.88), ('nao', 97.56), ('apen', 84.52), ('ped', 78.04)]
--------------------
Topic 1:
[('produt', 208.32), ('gost', 99.54), ('vei', 97.76), ('praz', 89.15), ('entreg', 85.59), ('compr', 78.59), ('defeit', 77.6), ('receb', 65.13), ('fot', 63.54), ('qual', 61.97)]
--------------------
Topic 2:
[('produt', 150.41), ('vei', 126.54), ('err', 104.63), ('compr', 96.41), ('entreg', 79.81), ('recom', 78.6), ('boa', 75.74), ('cheg', 74.7), ('demor', 73.72), ('cor', 65.84)]


🇧🇷 Burada biraz Brezilya Portekizcesi kelimeler var:
- _cadeiras = chairs_
- _produto = product_
- _recomendo = recommend (não recomendo == not recommend)_
- _bom = good_
- _comprei = bought_
- _veio = came_
- _errado = wrong_
- _gostaria = I would like to..._
- _duas = two_
- _nao = not_
- _entregue = delivered_
- _pecas = part_
- _ainda = yet_
- _recebi = received_

👉 Bir konuyla ilişkili en popüler kelimeleri göster

In [65]:
# 1. Her bir yorum için en baskın konuyu bul (0, 1 veya 2)
# doc_topic_mixture matrisindeki her satır için en yüksek olasılıklı indeksi alıyoruz
df_low["most_important_topic"] = doc_topic_mixture.argmax(axis=1)

# 2. Özellik isimlerini (kelimeleri) al
words = vectorizer.get_feature_names_out()

# 3. Her konu (Topic 0, 1, 2) için en önemli 5 kelimeyi içeren bir liste oluştur
topic_word_mixture = [
    ", ".join([words[i] for i in topic.argsort()[-5:][::-1]]) 
    for topic in lda_model.components_
]

In [89]:
df_low["most_important_words"] = df_low["most_important_topic"].apply(lambda i: topic_word_mixture[i])

In [102]:
def get_top_words(topic_idx, top_n=5):
    topic_probs = topic_word_mixture[topic_idx]
    top_indices = topic_probs.argsort()[-top_n:][::-1]
    return [feature_names[i] for i in top_indices]

df_low["most_important_words"] = df_low["most_important_topic"].apply(get_top_words)

In [103]:
df_low[["review_id",
        "review_score",
        "product_category_name",
        "cleaned_review",
        "most_important_topic",
        "most_important_words"]
      ].head()

,review_id,review_score,product_category_name,cleaned_review,most_important_topic,most_important_words
14,e233e51d11511bf30e568c76360ace52,1,eletronicos,receb soment control mide split estil falt con...,1,"[produt, gost, vei, praz, entreg]"
23,8b230a1373c6dc4bd867099fda1d7039,3,ferramentas_jardim,compr dua unidad receb agor faç,0,"[receb, bom, entreg, produt, aind]"
24,cb2fc3e5711b5ae85e0491ee18af63ed,3,beleza_saude,produt bom porém vei mim condiz fot anúnci,1,"[produt, gost, vei, praz, entreg]"
25,60c714ed14cef913944a3147094a4742,1,moveis_decoracao,produt inferi mal acab,1,"[produt, gost, vei, praz, entreg]"
28,0bd4dcc4f6c4621baf37f73495cad8c4,3,esporte_lazer,entreg praz,1,"[produt, gost, vei, praz, entreg]"


## (3.4) Pipeline Tf-Idf ve LDA

In [104]:
from sklearn import set_config
set_config("diagram")

🔨 Önceki Vectorizer ve LDA'yı birbirine bağlayan bir Pipeline oluşturun.

Temizlenmiş metinlere uyarlayın.

In [105]:
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ('tfidf', vectorizer),
    ('lda', lda_model)
])

doc_topic_mixture_pipe = pipe.transform(df_low['cleaned_review'].values.astype('U'))

print("Eğitilmiş nesneler Pipeline'a bağlandı ve transform başarıyla tamamlandı.")

Eğitilmiş nesneler Pipeline'a bağlandı ve transform başarıyla tamamlandı.


💡 `pipeline.components_` ile bileşenlere erişmeye çalışırsanız, Pipeline'da `components_` olmadığı için bu YÜRÜMEZ. Ancak, LDA'ya erişmek için `pipeline._final_estimator` kullanabilirsiniz. Ve buradan konulara erişebilirsiniz!

In [106]:
pipe._final_estimator

LatentDirichletAllocation(n_components=3, random_state=42)

In [107]:
pipe._final_estimator.components_

array([[ 0.86133639,  0.35257959,  0.3481354 , ...,  1.43147145,
         1.51887713,  3.40318385],
       [ 3.25939508,  2.74309222,  0.34188965, ...,  0.33435038,
         0.35039831,  0.42601774],
       [ 0.80337596, 10.95716107,  1.48277533, ...,  0.33401599,
         1.87343976,  0.35443908]])

Pipeline ile **Belge Karışımı**:

In [108]:
# Sonucu bir DataFrame yaparak daha okunabilir hale getirelim
import pandas as pd
column_names = [f"Konu_{i}" for i in range(pipe.named_steps['lda'].n_components)]
df_topics = pd.DataFrame(doc_topic_mixture_pipe, columns=column_names)

# Orijinal veriyle birleştirip ilk birkaç satıra bakalım
result_df = pd.concat([df_low[['review_score', 'cleaned_review']], df_topics], axis=1)
print(result_df.head())

    review_score                                     cleaned_review    Konu_0  \
14           1.0  receb soment control mide split estil falt con...  0.131818   
23           3.0                    compr dua unidad receb agor faç  0.114541   
24           3.0         produt bom porém vei mim condiz fot anúnci  0.112110   
25           1.0                             produt inferi mal acab  0.073303   
28           3.0                                        entreg praz  0.853992   

      Konu_1    Konu_2  
14  0.734002  0.134180  
23  0.589850  0.295609  
24  0.116248  0.771641  
25  0.095133  0.831564  
28  0.068539  0.077469  


Pipeline ile **Konu Karışımı**:

In [109]:
# 1. Pipeline'ın son adımı olan LDA modeline eriş
lda_model_from_pipe = pipe.named_steps['lda']

# 2. Konu-Kelime matrisini al (Topic-Word Mixture)
# Bu matris (n_components, n_features) boyutundadır.
topic_word_mixture = lda_model_from_pipe.components_

# 3. Anlamlı hale getirmek için özellik isimlerini (kelimeleri) çek
feature_names = pipe.named_steps['tfidf'].get_feature_names_out()

print(f"Konu Karışımı Matrisi Boyutu: {topic_word_mixture.shape}") # (3, 5000)

Konu Karışımı Matrisi Boyutu: (3, 5000)


## (4) 🎁 Ürün Kategorileri

### (4.1) Ürün kategorilerine göre gruplandırma

📈 Veri kümesini `product_category_name` ile gruplandırın ve performanslarını inceleyin.

In [110]:
# Performansa göre ürün kategorileri - sayı, ortalama, medyan ve standart sapmaya bakın
product_categories = df_low.groupby(by = 'product_category_name').agg({
        'review_score': ["count", "mean", "median", "std"]
    })

# Analiz için belirli bir süreden daha az satılan ürünleri kaldırma
cutoff = 50
product_categories = product_categories[product_categories[("review_score", "count")] > cutoff]

# Ürün kategorilerini performansa göre sıralama
product_categories = product_categories.sort_values(by = [('review_score', 'mean'),
                                                          ('review_score', 'std')],
                                                    ascending = [False, True])
product_categories

review_score                           
                                         count      mean median       std
product_category_name                                                    
consoles_games                              92  1.956522    2.0  0.924787
fashion_bolsas_e_acessorios                155  1.948387    2.0  0.917319
malas_acessorios                            66  1.924242    2.0  0.949666
papelaria                                  155  1.922581    2.0  0.922556
casa_construcao                             66  1.909091    2.0  0.836242
utilidades_domesticas                      586  1.889078    2.0  0.888473
cool_stuff                                 298  1.885906    2.0  0.906618
relogios_presentes                         589  1.881154    2.0  0.905088
pet_shop                                   147  1.857143    2.0  0.883641
moveis_escritorio                          261  1.854406    1.0  0.920843
cama_mesa_banho                           1221  1.850942    2.0  0.890235
esporte_lazer                              626  1.846645    2.0  0.899136
casa_conforto                               57  1.842105    2.0  0.902169
beleza_saude                               716  1.840782    1.0  0.908275
ferramentas_jardim                         318  1.817610    1.0  0.904635
telefonia                                  488  1.815574    2.0  0.875636
moveis_decoracao                           743  1.802153    1.0  0.888620
brinquedos                                 298  1.788591    1.0  0.890808
bebes                                      244  1.786885    1.0  0.877046
eletronicos                                248  1.778226    1.0  0.883502
construcao_ferramentas_construcao           71  1.774648    1.0  0.881511
automotivo                                 370  1.770270    1.0  0.870275
perfumaria                                 235  1.723404    1.0  0.879543
informatica_acessorios                     797  1.708908    1.0  0.867089
eletrodomesticos                            84  1.630952    1.0  0.875087
eletroportateis                             54  1.518519    1.0  0.770708

### (4.2) En kötü ürün kategorileri

👎 *Ortalama değerlendirme puanı* açısından en kötü beş kategoriyi `worst_products` adlı bir değişkene kaydedin.

In [116]:
worst_products = product_categories.tail(5).sort_values(by = [("review_score", "count")],
                                                       ascending = False)
worst_products

review_score                           
                              count      mean median       std
product_category_name                                         
informatica_acessorios          797  1.708908    1.0  0.867089
automotivo                      370  1.770270    1.0  0.870275
perfumaria                      235  1.723404    1.0  0.879543
eletrodomesticos                 84  1.630952    1.0  0.875087
eletroportateis                  54  1.518519    1.0  0.770708

👇 Yalnızca `worst_products` öğelerini içeren bir `worst_products_review` DataFrame oluşturun.

In [112]:
worst_products_reviews = df_low[df_low.product_category_name.isin(worst_products.index)]
worst_products_reviews[["review_id",
                        "review_score",
                        "product_category_name",
                        "cleaned_review",
                        "most_important_topic",
                        "most_important_words"]
      ]

,review_id,review_score,product_category_name,cleaned_review,most_important_topic,most_important_words
34,ff722b4c68783a4459a3adb9bb4e1d0d,3,informatica_acessorios,produt cheg pc consegu reconhec port usb,1,"[produt, gost, vei, praz, entreg]"
57,5f938e5f5f2e9a75710b54feeb9ea610,1,eletrodomesticos,médi peç serv,1,"[produt, gost, vei, praz, entreg]"
71,6b341682ab39af9fa00d72d7388c903b,3,automotivo,not compr produt corret cap intern outr,1,"[produt, gost, vei, praz, entreg]"
88,b736ff4204044e49e39062584d06fa74,1,eletroportateis,loj anunc produt entreg outr,2,"[produt, vei, err, compr, entreg]"
100,6f885e2fd69412264d9e74e805175d53,1,informatica_acessorios,receb bem rápid porém vei outr produt ped mode...,2,"[produt, vei, err, compr, entreg]"
...,...,...,...,...,...,...
36049,e9f0bde9a98ff79305964e9033ed8cd2,1,informatica_acessorios,produt cheg falt péss cond,2,"[produt, vei, err, compr, entreg]"
36054,44380245bf1a49875542ea0e7f216986,1,informatica_acessorios,receb produt aind assim receb produt avali posi,0,"[receb, bom, entreg, produt, aind]"
36095,970ae7c9df61910a0a30c38670db7ae5,3,informatica_acessorios,probl past multius pra notebook ped cor vermel...,2,"[produt, vei, err, compr, entreg]"
36100,504403288b387de1c95a193909ba5dfd,3,perfumaria,cheg bem rapid produt razoá,2,"[produt, vei, err, compr, entreg]"


### (4.3). En kötü ürünler için konular

❓ En kötü ürünlerin konuları nelerdir? ❓

In [113]:
worst_products_reviews["most_important_topic"].value_counts()

most_important_topic
0    607
2    484
1    449
Name: count, dtype: int64

In [114]:
bad_frequency = list(worst_products_reviews["most_important_topic"].value_counts().index)
bad_frequency

[0, 2, 1]

In [115]:
[topic_word_mixture[i] for i in bad_frequency]

[array([0.86133639, 0.35257959, 0.3481354 , ..., 1.43147145, 1.51887713,
        3.40318385]),
 array([ 0.80337596, 10.95716107,  1.48277533, ...,  0.33401599,
         1.87343976,  0.35443908]),
 array([3.25939508, 2.74309222, 0.34188965, ..., 0.33435038, 0.35039831,
        0.42601774])]

## (5) 🎁 Satıcılar...

* En kötü satıcılar tarafından ne tür ürünler satıldı?
* En kötü satıcılar için başlıca yorumlar nelerdir?

### (5.1) En kötü satıcılar

In [80]:
from olist.seller import Seller
sellers = Seller().get_training_data()
sellers.columns

Index(['seller_id', 'seller_city', 'seller_state', 'delay_to_carrier',
       'wait_time', 'date_first_sale', 'date_last_sale', 'months_on_olist',
       'n_orders', 'quantity', 'quantity_per_order', 'sales',
       'share_of_five_stars', 'share_of_one_stars', 'review_score',
       'cost_of_reviews', 'sales_fees', 'subscription_fees', 'revenues',
       'profits'],
      dtype='object')

👇 En kötü satan 10 ürünü seçin ve bunları `worst_sellers` adlı bir değişkene kaydedin.

In [81]:
worst_sellers = sellers[["seller_id", "review_score", "profits"]].sort_values(
    by = "profits",
    ascending = True).head(10)
worst_sellers

,seller_id,review_score,profits
769,6560211a19b47992c3666cc44a7e94c0,3.980674,-18769.517
2360,4a3ca9315b744ce9f8e9374361493884,3.853024,-15892.708
946,ea8482cd71df3c1969d7b9473ff13abc,4.026643,-14342.248
1358,cc419e0650a3c5ba77189a1882b7556a,4.155397,-13271.158
315,8b321bb669392f5163d04c59e235e066,4.104752,-11886.431
453,1f50f920176fa81dab994f9023523100,4.137956,-9586.079
1214,d2374cbcbb3ca4ab1086534108cc3ab7,3.766730,-8537.608
1133,7c67e1448b00f6e969d365cea6b010ab,3.498458,-7447.611
2689,1835b56ce799e6a4dc4eddc053f04066,3.665865,-6655.579
2025,cca3071e3e9bb7d12640c9fbe2301306,3.884943,-6429.011


### (5.2) En kötü satıcılar tarafından satılan ürünler

In [82]:
from olist.product import Product
products = Product().get_training_data() [["product_id", "category"]]
products

,product_id,category
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,art
2,96bd76ec8810374ed1b65e291975717f,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,baby
4,9dc1a7de274444849c219cff195d0b71,housewares
...,...,...
31479,a0b7d5a992ccda646f2d34e418fff5a0,furniture_decor
31480,bf4538d88321d0fd4412a93c974510e6,construction_tools_lights
31481,9a7c6041fa9592d9d9ef6cfe62a71f8c,bed_bath_table
31482,83808703fc0706a22e264b9d75f04a2e,computers_accessories


❓ En kötü satıcılar tarafından satılan ürün türleri nelerdir? ❓

In [83]:
from olist.data import Olist
data = Olist().get_data()
sellers_product_category = data["order_items"].merge(products,
                                             on = "product_id", how = "left")[["seller_id", "category"]]

sellers_product_category

,seller_id,category
0,48436dade18ac8b2bce089ec2a041202,cool_stuff
1,dd7ddc04e1b6c2c614352b383efe2d36,pet_shop
2,5b51032eddd242adc84c38acab88f23d,furniture_decor
3,9d7a1d34a5052409006425275ba1c2b4,perfumery
4,df560393f3a51e74553ab94004ba5c87,garden_tools
...,...,...
112645,b8bc237ba3788b23da09c0f1f3a3288c,housewares
112646,f3c38ab652836d21de61fb8314b69182,computers_accessories
112647,c3cfdc648177fdbbbb35635a37472c53,sports_leisure
112648,2b3e4a2a3ea8e01938cabda2a3e5cc79,computers_accessories


In [84]:
sellers_product_category.groupby("seller_id").count()

,category
seller_id,
0015a82c2db000af6aaaf3ae2ecb0532,3
001cca7ae9ae17fb1caed9dfb1094831,239
001e6ad469a905060d959994f1b41e4f,0
002100f778ceb8431b7a1020ff7ab48f,55
003554e2dce176b5555353e4f3555ac8,0
...,...
ffcfefa19b08742c5d315f2791395ee5,0
ffdd9f82b9a447f6f8d4b91554cc7dd3,20
ffeee66ac5d5a62fe688b9d26f83f534,14


### (5.3) En kötü satıcılar için kategoriler ve konular

🎁 İşte bazı kullanışlı işlevler:
- Bir satıcı tarafından satılan ürün kategorilerini göstermek için `focus_seller(seller_id)`
- Bir satıcı için en sık kullanılan konuların en popüler kelimelerini göstermek için `bad_reviews_seller`

In [85]:
def focus_seller(seller_id):
    return sellers_product_category[sellers_product_category.seller_id == seller_id].value_counts()

In [86]:
bad_reviews_sellers = worst_products_reviews.merge(data["order_items"])
bad_reviews_sellers.head(3)

,review_id,length_review,review_score,order_id,product_category_name,full_review,cleaned_review,most_important_topic,most_important_words,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,c422274e50e900b46fee429016c5458d,7,3,950bb159eea525ff8947b92c93b02f8e,moveis_escritorio,Gostei,gost,2,"produt, vei, err, compr, entreg",1,86f2416d4670e4ea3ca5494d043d9f24,7c67e1448b00f6e969d365cea6b010ab,2017-09-22 04:55:08,159.94,25.63
1,5d6f9cddc8335878d8bf20c3bd4602e8,162,5,38fc895ea0a2aa253a46a6fdbb65aaf5,moveis_escritorio,mega recomendo Recebi meu produto corretamente...,meg recom receb produt corret dentr praz verif...,2,"produt, vei, err, compr, entreg",1,2a5806f10d0f00e5ad032dd2e3c8806e,7c67e1448b00f6e969d365cea6b010ab,2018-08-09 10:24:11,169.99,34.27
2,1f836d160f50247a394034abd93b7c24,22,3,7951fe56cf685dca47524b1986bf56b6,casa_construcao,Esperando o instalador,esper instal,1,"produt, gost, vei, praz, entreg",1,5926c58c462b5d1f932ef3f0a9e37d41,c64a2aec32cc408a8a4c6d7c46017f91,2017-12-27 23:14:26,386.75,63.17


In [87]:
def bad_reviews_seller(bad_reviews_sellers, seller_id):
    mask = (bad_reviews_sellers.seller_id == seller_id)
    temp = bad_reviews_sellers[mask]
    if len(temp) > 0: # satıcı kötü yorumlar veri çerçevesinde görünüyorsa
        most_frequent_topic_seller = list(temp.most_important_topic.value_counts().head(1).index)[0]
        return topic_word_mixture[most_frequent_topic_seller]

❓Bu en az satan ürünlerin her biri için en sık kullanılan ürün kategorilerini ve kelimeleri gösterin ❓

In [88]:
for worst_seller in worst_sellers["seller_id"]:
    print("-"*50)
    print(f"Focusing on the seller #{worst_seller}...")
    print(focus_seller(worst_seller))
    print(bad_reviews_seller(bad_reviews_sellers, worst_seller))


--------------------------------------------------
Focusing on the seller #6560211a19b47992c3666cc44a7e94c0...
seller_id                         category                 
6560211a19b47992c3666cc44a7e94c0  watches_gifts                1624
                                  fashion_bags_accessories      340
                                  audio                          32
                                  perfumery                      13
                                  computers_accessories          12
                                  sports_leisure                  7
                                  construction_tools_safety       1
Name: count, dtype: int64
None
--------------------------------------------------
Focusing on the seller #4a3ca9315b744ce9f8e9374361493884...
seller_id                         category       
4a3ca9315b744ce9f8e9374361493884  bed_bath_table     1566
                                  home_confort        243
                                  furniture_d

🏁 Tebrikler. NLP'nin bazı temellerini (Ön İşleme + Vektörleştirme + NB/LDA) öğrendiniz ve bu yeni “uzmanlığı” Karar Bilimi ile birleştirdik.

💾 `git add / commit / push` yapmayı unutmayın.